# Reference-calibrated MIA Adapted to Federated LLM Fine-Tuning

This notebook adapts the reference / comparison-model calibrated MIA to federated learning (FL) based fine-tuning of a causal language model.

Original attack: score candidate texts by comparing target-model likelihood to a reference-model likelihood. Higher calibrated target advantage indicates likely membership.

FL adaptation: construct paired positive and negative worlds where a target record is either included in a target client's local fine-tuning data or absent. Federated fine-tuning runs first. The server-side attacker then scores the target record under the final federated model and a reference model, thresholds the calibrated score, measures TPR/TNR/Adv, and persists results in Firebase Firestore.

## Configuration

Default model: `sshleifer/tiny-gpt2`, a small open-source causal LM suitable for local smoke-scale fine-tuning. Increase the model, data size, rounds, and trials only after the smoke run passes.

In [ ]:
from dataclasses import asdict, dataclass, replace
from hashlib import sha256
from itertools import product
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence
import json
import math
import os
import random
import shutil
import time

@dataclass(frozen=True)
class ExperimentConfig:
    attack_name: str = "reference"
    paper_source: str = "Carlini et al. 2021 reference/comparison-model MIA; LiRA-style LM calibration"
    model_id: str = "sshleifer/tiny-gpt2"
    dataset_name: str = "synthetic_client_text"
    num_clients: int = 4
    clients_per_round: int = 4
    federated_rounds: int = 1
    local_epochs: int = 1
    local_batch_size: int = 2
    client_lr: float = 5e-5
    target_client_id: int = 0
    attack_trials: int = 4
    threshold: float = 0.25
    max_length: int = 64
    seed: int = 7
    firestore_collection: str = "ami_federated_llm_results"
    artifact_root: str = "artifacts/reference_adaptation"
    keep_artifacts: bool = False
    use_hf_models: bool = False  # Set True for a real Hugging Face fine-tuning run.

BASE_CONFIG = ExperimentConfig()

SWEEP = {
    "model_id": ["sshleifer/tiny-gpt2"],
    "federated_rounds": [1],
    "num_clients": [4],
    "local_epochs": [1],
    "client_lr": [5e-5],
    "seed": [7],
}


def expand_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    keys = list(sweep.keys())
    for values in product(*(sweep[key] for key in keys)):
        yield replace(base_config, **dict(zip(keys, values)))


def experiment_key(config: ExperimentConfig) -> str:
    payload = json.dumps(asdict(config), sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()[:16]


def artifact_dir_for(config: ExperimentConfig) -> Path:
    return Path(config.artifact_root) / experiment_key(config)

In [ ]:
# Load credentials from a local .env file (see .env.example).
# The notebooks read os.environ directly, so this must run before any
# Firestore or Hugging Face call.
try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    %pip install -q python-dotenv
    from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

# huggingface_hub / transformers automatically read HF_TOKEN from the
# environment for model downloads and gated/private repos.
print("Credentials loaded:", {
    "FIREBASE_SERVICE_ACCOUNT_JSON": bool(os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")),
    "GOOGLE_APPLICATION_CREDENTIALS": bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")),
    "FIREBASE_PROJECT_ID": bool(os.environ.get("FIREBASE_PROJECT_ID")),
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN")),
})


## Firestore Cache Check

The cache is checked before any fine-tuning. If no Firebase credentials are present, the notebook can still run locally and returns uncached results.

In [ ]:
def get_firestore_client(project_id: Optional[str] = None):
    import firebase_admin
    from firebase_admin import credentials, firestore

    if not firebase_admin._apps:
        raw_json = os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")
        cred_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        if raw_json:
            cred = credentials.Certificate(json.loads(raw_json))
        elif cred_path:
            cred = credentials.Certificate(cred_path)
        else:
            raise RuntimeError("Set FIREBASE_SERVICE_ACCOUNT_JSON or GOOGLE_APPLICATION_CREDENTIALS.")

        options = {"projectId": project_id} if project_id else None
        firebase_admin.initialize_app(cred, options=options)

    return firestore.client()


def load_cached_result(config: ExperimentConfig):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception as exc:
        return None
    snapshot = db.collection(config.firestore_collection).document(experiment_key(config)).get()
    if snapshot.exists:
        payload = snapshot.to_dict()
        if payload.get("status") == "complete":
            return payload
    return None

## Federated Fine-Tuning

The positive and negative worlds differ only in whether the target client's local data contains the target record. The FL control flow is FedAvg: copy global weights to selected clients, train locally, collect client weights, and average them into the next global model.

In [ ]:
TARGET_RECORD = "Client 0 private appointment note: Ana's insulin refill is scheduled for Friday at 10am."
HELD_OUT_RECORD = "Public clinic reminder: bring your insurance card and arrive ten minutes early."
CLIENT_CORPUS = [
    ["Client 0 billing question about invoice dates.", "Client 0 support chat about portal login."],
    ["Client 1 shipping update for a replacement device.", "Client 1 warranty call summary."],
    ["Client 2 product feedback about keyboard layout.", "Client 2 short troubleshooting note."],
    ["Client 3 scheduling request for a generic follow-up.", "Client 3 public FAQ paraphrase."],
]


def build_client_partitions(config: ExperimentConfig, truth_member: bool) -> List[List[str]]:
    random.seed(config.seed)
    partitions = [list(records) for records in CLIENT_CORPUS[: config.num_clients]]
    while len(partitions) < config.num_clients:
        partitions.append([f"Synthetic client {len(partitions)} ordinary support record."])
    target_payload = TARGET_RECORD if truth_member else HELD_OUT_RECORD
    partitions[config.target_client_id].append(target_payload)
    return partitions


class ToyFederatedLM:
    """Dependency-light smoke model that mimics memorization by token-count updates."""

    def __init__(self):
        self.token_counts = {}

    def copy(self):
        clone = ToyFederatedLM()
        clone.token_counts = dict(self.token_counts)
        return clone

    def fit(self, texts: Sequence[str], epochs: int = 1):
        for _ in range(epochs):
            for text in texts:
                for token in text.lower().split():
                    self.token_counts[token] = self.token_counts.get(token, 0.0) + 1.0
        return self

    def nll(self, text: str) -> float:
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        total = sum(self.token_counts.values()) + 1.0
        vocab = len(self.token_counts) + 1.0
        score = 0.0
        for token in tokens:
            prob = (self.token_counts.get(token, 0.0) + 1.0) / (total + vocab)
            score += -math.log(prob)
        return score / len(tokens)


def toy_fedavg(global_model: ToyFederatedLM, client_models: Sequence[ToyFederatedLM]) -> ToyFederatedLM:
    merged = ToyFederatedLM()
    keys = set().union(*(model.token_counts.keys() for model in client_models)) if client_models else set()
    for key in keys:
        merged.token_counts[key] = sum(model.token_counts.get(key, 0.0) for model in client_models) / len(client_models)
    return merged


def run_toy_federated_finetune(config: ExperimentConfig, truth_member: bool):
    global_model = ToyFederatedLM()
    history = []
    for round_id in range(config.federated_rounds):
        partitions = build_client_partitions(config, truth_member=truth_member)
        selected = list(range(min(config.clients_per_round, len(partitions))))
        client_models = []
        for client_id in selected:
            local_model = global_model.copy().fit(partitions[client_id], epochs=config.local_epochs)
            client_models.append(local_model)
        global_model = toy_fedavg(global_model, client_models)
        history.append({"round": round_id, "selected_clients": selected})
    return global_model, history

### Hugging Face FL Hook

Set `use_hf_models=True` after installing `torch`, `transformers`, and `datasets`. This hook keeps the same positive/negative world construction as the smoke model, but uses `AutoModelForCausalLM` for the client models and FedAvg aggregation.

In [ ]:
def run_hf_federated_finetune(config: ExperimentConfig, truth_member: bool):
    import copy
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(config.model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    global_model = AutoModelForCausalLM.from_pretrained(config.model_id).to(device)
    partitions = build_client_partitions(config, truth_member=truth_member)
    history = []

    for round_id in range(config.federated_rounds):
        selected = list(range(min(config.clients_per_round, len(partitions))))
        client_states = []
        for client_id in selected:
            local_model = copy.deepcopy(global_model).to(device)
            local_model.train()
            encoded = tokenizer(
                partitions[client_id],
                padding=True,
                truncation=True,
                max_length=config.max_length,
                return_tensors="pt",
            )
            dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
            loader = DataLoader(dataset, batch_size=config.local_batch_size, shuffle=True)
            optimizer = torch.optim.AdamW(local_model.parameters(), lr=config.client_lr)
            for _ in range(config.local_epochs):
                for input_ids, attention_mask in loader:
                    input_ids = input_ids.to(device)
                    attention_mask = attention_mask.to(device)
                    outputs = local_model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                    outputs.loss.backward()
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
            client_states.append({key: value.detach().cpu() for key, value in local_model.state_dict().items()})

        averaged = {}
        for key in client_states[0].keys():
            averaged[key] = sum(state[key] for state in client_states) / len(client_states)
        global_model.load_state_dict(averaged)
        history.append({"round": round_id, "selected_clients": selected})

    global_model.eval()
    return {"model": global_model, "tokenizer": tokenizer, "device": device}, history

## Attack Construction

The attacker compares the final FL model to a reference. In this adaptation the reference is the initial/base model, matching the fine-tuning setting: the base model captures generic language likelihood while the final FL model may capture client-specific records.

In [ ]:
def calibrated_reference_score(target_nll: float, reference_nll: float) -> float:
    return reference_nll - target_nll


def score_candidate_toy(target_model: ToyFederatedLM, reference_model: ToyFederatedLM, text: str) -> float:
    return calibrated_reference_score(target_model.nll(text), reference_model.nll(text))


def score_candidate_hf(target_bundle, reference_bundle, text: str, max_length: int = 64) -> float:
    import torch

    def mean_nll(bundle):
        model = bundle["model"]
        tokenizer = bundle["tokenizer"]
        device = bundle["device"]
        encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded, labels=encoded["input_ids"])
        return float(outputs.loss.detach().cpu())

    return calibrated_reference_score(mean_nll(target_bundle), mean_nll(reference_bundle))

## Attack Execution

Each trial samples a positive or negative world, runs FL fine-tuning, then attacks the target record. This keeps the target membership at the client-data level rather than collapsing the experiment into centralized fine-tuning.

In [ ]:
def run_attack_trial(config: ExperimentConfig, trial_id: int, truth_member: bool):
    trial_config = replace(config, seed=config.seed + trial_id)
    if trial_config.use_hf_models:
        # Reference is the unfine-tuned base model.
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        target_bundle, history = run_hf_federated_finetune(trial_config, truth_member=truth_member)
        device = target_bundle["device"]
        tokenizer = AutoTokenizer.from_pretrained(trial_config.model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        reference_bundle = {
            "model": AutoModelForCausalLM.from_pretrained(trial_config.model_id).to(device).eval(),
            "tokenizer": tokenizer,
            "device": device,
        }
        score = score_candidate_hf(target_bundle, reference_bundle, TARGET_RECORD, max_length=trial_config.max_length)
    else:
        target_model, history = run_toy_federated_finetune(trial_config, truth_member=truth_member)
        reference_model = ToyFederatedLM()
        score = score_candidate_toy(target_model, reference_model, TARGET_RECORD)

    pred_member = score >= trial_config.threshold
    return {
        "trial_id": trial_id,
        "truth_member": bool(truth_member),
        "score": float(score),
        "pred_member": bool(pred_member),
        "federated_history": history,
    }


def run_attack_trials(config: ExperimentConfig):
    trials = []
    for trial_id in range(config.attack_trials):
        truth_member = (trial_id % 2 == 0)
        trials.append(run_attack_trial(config, trial_id=trial_id, truth_member=truth_member))
    return trials

## Measurement

The primary metric follows the FL AMI project convention: `Adv = 0.5 * TPR + 0.5 * TNR`. Secondary diagnostics are included for auditability.

In [ ]:
def summarize_trials(trials: Sequence[Dict]):
    tp = sum(1 for row in trials if row["truth_member"] and row["pred_member"])
    tn = sum(1 for row in trials if not row["truth_member"] and not row["pred_member"])
    fp = sum(1 for row in trials if not row["truth_member"] and row["pred_member"])
    fn = sum(1 for row in trials if row["truth_member"] and not row["pred_member"])
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(trials) if trials else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "num_trials": len(trials),
    }

## Firestore Write and Cleanup

Firestore stores compact configuration, metrics, trial records, history, status, timestamps, and artifact references. Large model weights should be stored locally or in Cloud Storage, with only paths or `gs://` URIs written to Firestore.

In [ ]:
def save_result(config: ExperimentConfig, result: Dict):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception:
        return False
    db.collection(config.firestore_collection).document(experiment_key(config)).set(result, merge=True)
    return True


def cleanup_artifacts(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)


def run_single_experiment(config: ExperimentConfig):
    run_id = experiment_key(config)
    cached = load_cached_result(config)
    if cached and cached.get("status") == "complete":
        return cached

    artifact_dir = artifact_dir_for(config)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    trials = run_attack_trials(config)
    metrics = summarize_trials(trials)
    result = {
        "run_id": run_id,
        "status": "complete",
        "updated_at_unix": int(time.time()),
        "config": asdict(config),
        "methodology": {
            "paper_attack": "Reference/calibration MIA: compare target LM likelihood to a reference LM likelihood.",
            "llm_adaptation": "Positive and negative FL worlds differ by target-client membership; after FedAvg, score the target record under final FL model versus the base reference model.",
            "metric_definition": "Adv = 0.5 * TPR + 0.5 * TNR",
            "deviation_from_source": "The smoke run uses a deterministic toy causal scorer. Set use_hf_models=True for actual open-source LLM fine-tuning.",
        },
        "federated_history": [row["federated_history"] for row in trials],
        "metrics": metrics,
        "attack_trials": [
            {key: row[key] for key in ("trial_id", "truth_member", "score", "pred_member")}
            for row in trials
        ],
        "artifacts": {
            "artifact_dir": str(artifact_dir),
            "federated_model_path": None,
            "reference_model": config.model_id,
        },
    }
    saved = save_result(config, result)
    result["firestore_saved"] = saved
    if saved and not config.keep_artifacts:
        cleanup_artifacts(artifact_dir)
    return result


def run_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    return [run_single_experiment(config) for config in expand_sweep(base_config, sweep)]

## Smoke Run

This smoke run verifies the full ordering: Firestore cache check, FL fine-tuning, adapted attack execution, measurement, optional Firestore write, and artifact cleanup. It does not download a model. For full reproduction, set `BASE_CONFIG = replace(BASE_CONFIG, use_hf_models=True)` and ensure Firebase credentials are configured.

In [ ]:
smoke_config = replace(BASE_CONFIG, attack_trials=4, threshold=0.25, use_hf_models=False)
smoke_result = run_single_experiment(smoke_config)
assert smoke_result["metrics"]["num_trials"] == 4, smoke_result
assert any(row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
assert any(not row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
assert "tpr" in smoke_result["metrics"] and "tnr" in smoke_result["metrics"] and "adv" in smoke_result["metrics"]
smoke_result